# Neo4j Simple Connection Test

Validates connectivity and creates the UC JDBC connection.

**Prerequisites:**
- Run `./getting-started/upload_data.sh` (from the repo root) to upload CSV files to the UC Volume
- Run `./create_secrets.sh` (from the repo root) to store notebook configuration as Databricks secrets
- Run `00-load-graph.ipynb` to load the aircraft graph into Neo4j

## Configuration

In [ ]:
# =============================================================================
# CONFIGURATION - Loaded from Databricks secrets
# =============================================================================

# --- Neo4j Aura ---
SECRET_SCOPE = "neo4j-uc-demos"
NEO4J_URI = dbutils.secrets.get(scope=SECRET_SCOPE, key="NEO4J_URI")
NEO4J_USERNAME = dbutils.secrets.get(scope=SECRET_SCOPE, key="NEO4J_USERNAME")
NEO4J_PASSWORD = dbutils.secrets.get(scope=SECRET_SCOPE, key="NEO4J_PASSWORD")

# --- Databricks Unity Catalog ---
UC_CATALOG = dbutils.secrets.get(scope=SECRET_SCOPE, key="UC_CATALOG")
UC_SCHEMA = dbutils.secrets.get(scope=SECRET_SCOPE, key="UC_SCHEMA")
UC_VOLUME = dbutils.secrets.get(scope=SECRET_SCOPE, key="UC_VOLUME")
JDBC_JAR_PATH = dbutils.secrets.get(scope=SECRET_SCOPE, key="JDBC_JAR_PATH")
UC_CONNECTION_NAME = "sample_neo4j_jdbc_connection"

# =============================================================================
# DERIVED VALUES - no need to edit below this line
# =============================================================================
FQN = f"`{UC_CATALOG}`.`{UC_SCHEMA}`"
VOLUME_PATH = f"/Volumes/{UC_CATALOG}/{UC_SCHEMA}/{UC_VOLUME}"
NEO4J_JDBC_URL_SQL = f"jdbc:{NEO4J_URI}/neo4j?enableSQLTranslation=true"
JAVA_DEPENDENCIES = f'["{JDBC_JAR_PATH}"]'

print("Configuration:")
print(f"  Neo4j URI:       {NEO4J_URI}")
print(f"  Tables:          {FQN}.*")
print(f"  Volume:          {VOLUME_PATH}")
print(f"  JDBC JAR:        {JDBC_JAR_PATH}")
print(f"  UC Connection:   {UC_CONNECTION_NAME}")

---

## Section 1: Create UC JDBC Connection

Creates the Unity Catalog JDBC connection for SQL-to-Cypher federation.
The SafeSpark sandbox wraps the JDBC driver in an isolated JVM.

**Note:** `java_dependencies` only accepts UC Volume paths.

In [ ]:
import time

print("--- Section 1: Create UC JDBC Connection ---")
print(f"  Connection: {UC_CONNECTION_NAME}")
print(f"  URL:        {NEO4J_JDBC_URL_SQL}")

spark.sql(f"DROP CONNECTION IF EXISTS {UC_CONNECTION_NAME}")

esc = lambda s: s.replace("'", "\\'")

create_sql = f"""
    CREATE CONNECTION {UC_CONNECTION_NAME} TYPE JDBC
    ENVIRONMENT (
        java_dependencies '{JAVA_DEPENDENCIES}'
    )
    OPTIONS (
        url '{esc(NEO4J_JDBC_URL_SQL)}',
        user '{esc(NEO4J_USERNAME)}',
        password '{esc(NEO4J_PASSWORD)}',
        driver 'org.neo4j.jdbc.Neo4jDriver',
        externalOptionsAllowList 'dbtable,query,partitionColumn,lowerBound,upperBound,numPartitions,fetchSize,customSchema'
    )
"""

start = time.time()
spark.sql(create_sql)
elapsed = (time.time() - start) * 1000

df = (
    spark.read.format("jdbc")
    .option("databricks.connection", UC_CONNECTION_NAME)
    .option("query", "SELECT 1 AS test")
    .option("customSchema", "test INT")
    .load()
)
test_val = df.collect()[0]["test"]
status = "PASS" if test_val == 1 else "FAIL"
print(f"  [PASS] Connection created in {elapsed:.0f}ms")
print(f"  [{status}] SELECT 1 returned {test_val}")

---

## Section 2: Query via UC JDBC

Runs SQL queries against the Neo4j graph through the Unity Catalog JDBC connection.
SQL is automatically translated to Cypher by the connector.

**Note:** `customSchema` is required because Neo4j's JDBC driver returns `NullType` during Spark's schema inference.

In [ ]:
print("--- Section 2: Query via UC JDBC ---")

def read_neo4j(custom_schema, query):
    return (
        spark.read.format("jdbc")
        .option("databricks.connection", UC_CONNECTION_NAME)
        .option("customSchema", custom_schema)
        .option("query", query)
        .load()
    )

# Aircraft count
df = read_neo4j("aircraft_count LONG", "SELECT COUNT(*) AS aircraft_count FROM Aircraft")
count = df.collect()[0]["aircraft_count"]
status = "PASS" if count == 20 else "FAIL"
print(f"  [{status}] Aircraft: {count}")

# Airport count
df = read_neo4j("airport_count LONG", "SELECT COUNT(*) AS airport_count FROM Airport")
count = df.collect()[0]["airport_count"]
status = "PASS" if count == 12 else "FAIL"
print(f"  [{status}] Airports: {count}")

# Flights by operator
print("\n  Flights by operator:")
read_neo4j(
    "operator STRING, flight_count LONG",
    "SELECT operator, COUNT(*) AS flight_count FROM Flight GROUP BY operator ORDER BY flight_count DESC",
).show(truncate=False)

print("Status: PASS. Run 02-federated-queries.ipynb next")